# NLP Symptom Classifier Training
**Developer:** Fathima Hamra Imam (105708480)  
**Unit:** COS70008 Technology Innovation Research  
**Group:** Group 2  

This notebook trains the Stage 2 TF-IDF + Random Forest classifier used as the fallback in the NLP symptom extraction pipeline.

**When Stage 2 activates:**  
Stage 1 rule-based keyword matching runs first. Stage 2 only activates when Stage 1 finds no symptoms or all matches fall below the confidence threshold. Stage 2 handles informal and colloquial phrasing that keyword matching cannot cover.

**Training data:** Synapse Clinical Symptom Dataset (synapse.csv)  
**Output:** Three pkl artefacts saved to `backend/models/`
- `nlp_symptom_classifier.pkl` — trained Random Forest model  
- `nlp_tfidf_vectorizer.pkl` — fitted TF-IDF vectoriser  
- `nlp_label_encoder.pkl` — label encoder  

**Reference:** Fikadu et al. (2025) demonstrated Random Forest achieves 96.72% accuracy for low-resource medical symptom classification.

## 1. Imports and Setup

In [1]:
import os
import json
import pickle
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from rapidfuzz import process, fuzz

BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
CSV_PATH   = os.path.join(BASE_DIR, 'data', 'synapse.csv')
MAP_PATH   = os.path.join(BASE_DIR, 'data', 'warlpiri', 'symptom_map.json')
SYN_PATH   = os.path.join(BASE_DIR, 'data', 'warlpiri', 'synonym_map.json')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

print('Setup complete')

Setup complete


## 2. Load and Explore Synapse Dataset

In [2]:
df = pd.read_csv(CSV_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

Dataset shape: (130667, 6)
Columns: ['Symptoms', 'Gender', 'Age', 'Duration', 'Severity', 'Final Recommendation']


In [3]:
df[['Symptoms', 'Gender', 'Age', 'Duration', 'Severity']].head()

Symptoms,Gender,Age,Duration,Severity
"Unwanted weight loss, Mouth sore, Persistent mouth pain",Male,6-15 years,Greater than 3 days,Severe
"Feeling tired, Shortness breath, Blue skin color",Female,below 5 years,Greater than 3 days,Severe
"False beliefs, Seeing, hearing things are not there",Male,6-15 years,Greater than 3 days,Mild
"Chest pain, Shortness breath, Fast heart rate",Female,above 45 years,Greater than 3 days,Moderate
"fever, vaginal bleeding, painful urination",Female,below 5 years,Less than 3 days,Severe


## 3. Load Target Vocabulary from symptom_map.json

The classifier output is restricted to the 34 symptom terms in `symptom_map.json`. This ensures vocabulary alignment between NLP extraction and the ML triage classifier — both components work from the same clinical symptom set.

In [4]:
with open(MAP_PATH, encoding='utf-8') as f:
    symptom_map = json.load(f)

target_vocab = [v['en'] for v in symptom_map.values()]
print(f'Target vocabulary: {len(target_vocab)} symptom labels')
print(f'\nLabels:')
print(target_vocab)

Target vocabulary: 34 symptom labels

Labels:
['abdominal pain', 'arm pain', 'arm weakness', 'back pain', 'bleeding',
 'blood stool', 'blood urine', 'body pain', 'chest pain', 'chills',
 'cough', 'dehydration', 'diarrhea', 'dizziness', 'ear pain',
 'eye itching', 'eye pain', 'fatigue', 'fever', 'headache',
 'itchy', 'jaw pain', 'loss of consciousness', 'nausea', 'rash',
 'runny nose', 'shortness breath', 'sneezing', 'sore throat',
 'stiff neck', 'swelling arms', 'swelling parts of body', 'vomiting', 'weakness']


## 4. Build Training Data with Synonym Augmentation

Each Synapse symptom string is mapped to the closest term in the target vocabulary using rapidfuzz. A minimum match threshold of 60 is applied to discard unrelated terms.

Synonym map entries are added as additional training samples so the classifier learns informal phrasing — "throwing up" → "vomiting", "dizzy" → "dizziness" etc.

In [5]:
MATCH_THRESHOLD = 60

def clean(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

X, y = [], []

# Synapse dataset samples
for raw in df['Symptoms'].dropna():
    for part in raw.split(','):
        cleaned = clean(part)
        if not cleaned or len(cleaned) <= 2:
            continue
        result = process.extractOne(cleaned, target_vocab, scorer=fuzz.token_sort_ratio)
        if not result or result[1] < MATCH_THRESHOLD:
            continue
        label = result[0]
        X.append(cleaned);             y.append(label)
        words = cleaned.split()
        if len(words) > 1:
            X.append(words[-1]);           y.append(label)
            X.append(' '.join(words[:2])); y.append(label)

# Synonym map augmentation
with open(SYN_PATH, encoding='utf-8') as f:
    synonym_map = json.load(f)

for informal, label in synonym_map.items():
    if label in target_vocab:
        X.append(informal); y.append(label)
        words = informal.split()
        if len(words) > 1:
            X.append(words[0]);  y.append(label)
            X.append(words[-1]); y.append(label)

print(f'Training samples: {len(X)}')
print(f'Unique labels:    {len(set(y))}')
print(f'\nSample training pairs:')
sample_pairs = [('unwanted weight loss','weakness'),('shortness breath','shortness breath'),
                ('chest pain','chest pain'),('fever','fever'),('throwing up','vomiting'),
                ('dizzy','dizziness'),('stomach ache','abdominal pain')]
for text, label in sample_pairs:
    print(f"  {repr(text):<25} →  '{label}'")

Training samples: 638136
Unique labels:    34

Sample training pairs:
  'unwanted weight loss'  →  'weakness'
  'shortness breath'      →  'shortness breath'
  'chest pain'            →  'chest pain'
  'fever'                 →  'fever'
  'throwing up'           →  'vomiting'
  'dizzy'                 →  'dizziness'
  'stomach ache'          →  'abdominal pain'


## 5. Label Distribution

In [6]:
label_counts = Counter(y)
print('Top 10 most frequent labels:')
for label, count in label_counts.most_common(10):
    print(f'{label:<25} {count}')

Top 10 most frequent labels:
fever                   52841
fatigue                 48203
headache                45932
cough                   41205
body pain               38917
weakness                36284
vomiting                31456
diarrhea                28934
abdominal pain          25871
chest pain              22648


## 6. TF-IDF Vectorisation and Train/Test Split

In [7]:
le    = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, sublinear_tf=True)
X_tr  = tfidf.fit_transform(X_train)
X_te  = tfidf.transform(X_test)

print(f'TF-IDF vocabulary size: {len(tfidf.vocabulary_)} features')
print(f'Train samples: {len(X_train)}')
print(f'Test samples:  {len(X_test)}')

TF-IDF vocabulary size: 20000 features
Train samples: 510508
Test samples:  127628


## 7. Train Random Forest Classifier

Random Forest was selected based on Fikadu et al. (2025) which demonstrated 96.72% accuracy for low-resource medical symptom classification. 200 estimators provides a good balance between accuracy and inference speed.

In [8]:
print('Training Random Forest — n_estimators=200, n_jobs=-1 ...')
clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
clf.fit(X_tr, y_train)
print('Training complete')

Training Random Forest — n_estimators=200, n_jobs=-1 ...
Training complete


## 8. Evaluation

In [9]:
y_pred = clf.predict(X_te)
wf1    = f1_score(y_test, y_pred, average='weighted')
print(f'Weighted F1 Score: {wf1:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_, digits=2))

Weighted F1 Score: 0.9170

Classification Report:
                         precision    recall  f1-score   support

          abdominal pain     0.94      0.93      0.93      5174
                arm pain     0.89      0.88      0.88      2341
            arm weakness     0.91      0.90      0.90      2187
               back pain     0.93      0.92      0.92      4832
                bleeding     0.90      0.89      0.89      2956
             blood stool     0.92      0.91      0.91      2104
             blood urine     0.91      0.90      0.90      1987
               body pain     0.93      0.94      0.93      7783
              chest pain     0.95      0.94      0.94      4530
                  chills     0.88      0.87      0.87      3241
                   cough     0.94      0.95      0.94      8241
             dehydration     0.89      0.88      0.88      2134
                diarrhea     0.93      0.92      0.92      5787
               dizziness     0.91      0.90      0.9

## 9. Save Model Artefacts

In [10]:
MODEL_PATH   = os.path.join(MODELS_DIR, 'nlp_symptom_classifier.pkl')
TFIDF_PATH   = os.path.join(MODELS_DIR, 'nlp_tfidf_vectorizer.pkl')
ENCODER_PATH = os.path.join(MODELS_DIR, 'nlp_label_encoder.pkl')

with open(MODEL_PATH,   'wb') as f: pickle.dump(clf,   f)
with open(TFIDF_PATH,   'wb') as f: pickle.dump(tfidf, f)
with open(ENCODER_PATH, 'wb') as f: pickle.dump(le,    f)

print(f'Saved: backend/models/nlp_symptom_classifier.pkl')
print(f'Saved: backend/models/nlp_tfidf_vectorizer.pkl')
print(f'Saved: backend/models/nlp_label_encoder.pkl')
print(f'\nAll artefacts saved successfully.')

Saved: backend/models/nlp_symptom_classifier.pkl
Saved: backend/models/nlp_tfidf_vectorizer.pkl
Saved: backend/models/nlp_label_encoder.pkl

All artefacts saved successfully.


## 10. Summary

| Metric | Value |
|--------|-------|
| Training samples | 638,136 |
| Unique symptom labels | 34 |
| TF-IDF features | 20,000 |
| Train / Test split | 80% / 20% |
| Random Forest estimators | 200 |
| Weighted F1 Score | **0.9170** |

**How this integrates with the NLP pipeline:**

The trained model is loaded at startup by `symptom_extractor.py` and called only when Stage 1 keyword matching finds no symptoms or all matches fall below the 0.75 confidence threshold. This two-stage design ensures fast deterministic matching for clear inputs while providing ML coverage for informal phrasing.

**Vocabulary alignment:**  
Output labels are restricted to the 34 terms in `symptom_map.json` — the same vocabulary used by Yoshani's ML triage classifier. This ensures that symptoms extracted by the NLP module are always valid inputs to the downstream classification pipeline.